In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("..")  # so the notebook can import from src/

import pandas as pd
import numpy as np
import lightgbm
import sklearn

from src import config
from src.evaluate import evaluate

print("All imports OK")
print("Looking for data in:", config.RAW_DIR)

All imports OK
Looking for data in: D:\Projects\Main Projects\fraud-detection\data\raw


# **Loading Merged Datasets**

In [3]:
from src.data import load_processed
df = load_processed()
print(df.shape)
print(df.head())

(590540, 434)
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ...                id_31  id_32  \
0    NaN  150.0    discover  142.0  ...                  NaN    NaN   
1  404.0  150.0  mastercard  102.0  ...                  NaN    NaN   
2  490.0  150.0        visa  166.0  ...                  NaN    NaN   
3  567.0  150.0  mastercard  117.0  ...                  NaN    NaN   
4  514.0  150.0  mastercard  102.0  ...  samsung browser 6.2   32.0   

       id_33           id_34  id_35 id_36 id_37  id_38  DeviceType  \
0        NaN             N

# **Time-Based Split**

In [4]:
from src import config

# 1. Sort by transaction time — earliest first
df_sorted = df.sort_values(config.TIME_COL).reset_index(drop=True)

# 2. Cut at the 80% mark in time: earliest 80% -> train, latest 20% -> test
cutoff = int(len(df_sorted) * 0.80)

train_df = df_sorted.iloc[:cutoff]
test_df  = df_sorted.iloc[cutoff:]

# 3. Sanity checks
print("Train rows:", len(train_df), "| Test rows:", len(test_df))
print("Train time range:", train_df[config.TIME_COL].min(), "->", train_df[config.TIME_COL].max())
print("Test time range: ", test_df[config.TIME_COL].min(), "->", test_df[config.TIME_COL].max())
print("Train fraud rate:", round(train_df[config.TARGET].mean(), 4))
print("Test fraud rate: ", round(test_df[config.TARGET].mean(), 4))

Train rows: 472432 | Test rows: 118108
Train time range: 86400 -> 12192842
Test time range:  12192900 -> 15811131
Train fraud rate: 0.0351
Test fraud rate:  0.0344


# **Pre-processing**

In [5]:
# Separate features and target, drop columns that shouldn't be features
drop_cols = [config.TARGET, config.ID_COL, config.TIME_COL]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df[config.TARGET]
X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df[config.TARGET]

# Identify categorical (object/string) columns
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
print(f"{len(cat_cols)} categorical columns")
print(cat_cols)

31 categorical columns
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


C:\Users\Dell\AppData\Local\Temp\ipykernel_28640\873821224.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()


# **LR Baseline With Pipeline**

In [6]:
for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col]  = X_test[col].astype('category')

In [7]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from src.evaluate import evaluate

# Work on fresh copies so we don't disturb the LightGBM-ready frames
Xtr = X_train.copy()
Xte = X_test.copy()

# --- Decision 1: drop categoricals >90% missing (only for LR) ---
cat_missing = Xtr[cat_cols].isna().mean()
cat_keep = [c for c in cat_cols if cat_missing[c] <= 0.90]
cat_drop = [c for c in cat_cols if cat_missing[c] > 0.90]
print(f"Keeping {len(cat_keep)} categoricals, dropping {len(cat_drop)} (>90% missing): {cat_drop}")

# Numeric columns = everything not categorical
num_cols = [c for c in Xtr.columns if c not in cat_cols]
print(f"{len(num_cols)} numeric columns")

# Categorical dtype confuses the OHE pipeline; cast kept cats back to object
for c in cat_keep:
    Xtr[c] = Xtr[c].astype('object')
    Xte[c] = Xte[c].astype('object')

# --- Preprocessing pipelines ---
numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', max_categories=20, sparse_output=True)),
])

preprocess = ColumnTransformer([
    ('num', numeric_pipe, num_cols),
    ('cat', categorical_pipe, cat_keep),
])

# --- Full LR pipeline ---
lr = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',   # handles the 3.5% imbalance
        n_jobs=-1,
    )),
])

print("Fitting LR baseline... (this will take a few minutes)")
lr.fit(Xtr, y_train)

# Predict probabilities for the fraud class
y_prob_lr = lr.predict_proba(Xte)[:, 1]

print("\n=== Logistic Regression baseline ===")
evaluate(y_test, y_prob_lr)

Keeping 29 categoricals, dropping 2 (>90% missing): ['id_23', 'id_27']
400 numeric columns
Fitting LR baseline... (this will take a few minutes)


d:\Projects\Main Projects\fraud-detection\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



=== Logistic Regression baseline ===
Threshold: 0.5
Precision: 0.1241
Recall:    0.6978
F1:        0.2107
PR-AUC:    0.1856

Confusion matrix [[TN FP] [FN TP]]:
[[94021 20023]
 [ 1228  2836]]

               precision    recall  f1-score   support

           0     0.9871    0.8244    0.8985    114044
           1     0.1241    0.6978    0.2107      4064

    accuracy                         0.8201    118108
   macro avg     0.5556    0.7611    0.5546    118108
weighted avg     0.9574    0.8201    0.8748    118108



0.18561295419271878

# **LightGBM**

In [10]:
import lightgbm as lgb
from src.evaluate import evaluate

spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {spw:.1f}")

model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    random_state=config.RANDOM_STATE,
    n_jobs=-1,
)

# Train the full 500 trees — no early stopping, no eval_set
model.fit(X_train, y_train)

y_prob_lgb = model.predict_proba(X_test)[:, 1]

print("Best iteration:", model.best_iteration_, "| Trees:", model.n_estimators_)
print("Prob range:", y_prob_lgb.min().round(4), "-", y_prob_lgb.max().round(4))

print("\n=== LightGBM (full 500 trees) ===")
evaluate(y_test, y_prob_lgb)

scale_pos_weight = 27.5
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.933604 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34682
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 430
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784
Best iteration: 0 | Trees: 500
Prob range: 0.0 - 0.9999

=== LightGBM (full 500 trees) ===
Threshold: 0.5
Precision: 0.3295
Recall:    0.6572
F1:        0.4390
PR-AUC:    0.5484

Conf

0.5484036288561909

In [11]:
import numpy as np

print(f"{'thresh':>8} {'precision':>10} {'recall':>8} {'F1':>8} {'flagged':>9}")
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pred = (y_prob_lgb >= t).astype(int)
    n_flagged = pred.sum()
    tp = ((pred == 1) & (y_test.values == 1)).sum()
    prec = tp / n_flagged if n_flagged > 0 else 0
    rec = tp / (y_test.values == 1).sum()
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    print(f"{t:>8.2f} {prec:>10.4f} {rec:>8.4f} {f1:>8.4f} {n_flagged:>9}")

  thresh  precision   recall       F1   flagged
    0.30     0.1934   0.7822   0.3101     16437
    0.40     0.2564   0.7219   0.3784     11443
    0.50     0.3295   0.6572   0.4390      8105
    0.60     0.4231   0.5893   0.4926      5660
    0.70     0.5468   0.5145   0.5302      3824
    0.80     0.6805   0.4277   0.5252      2554
    0.90     0.8195   0.3273   0.4677      1623


In [12]:
OPERATING_THRESHOLD = 0.50

In [15]:
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from src.evaluate import evaluate

# --- Prep: SMOTE needs numeric, NaN-free data ---
# Work on copies so we don't disturb the originals
Xtr_s = X_train.copy()
Xte_s = X_test.copy()

# 1. Label-encode categoricals to numeric (approximate — treats cats as numbers)
for col in cat_cols:
    le = LabelEncoder()
    # Combine train+test categories so encoding is consistent, fill NaN with a sentinel
    combined = pd.concat([Xtr_s[col], Xte_s[col]]).astype(str).fillna('missing')
    le.fit(combined)
    Xtr_s[col] = le.transform(Xtr_s[col].astype(str).fillna('missing'))
    Xte_s[col] = le.transform(Xte_s[col].astype(str).fillna('missing'))

# 2. Fill remaining NaNs in numeric columns (SMOTE can't handle NaN)
Xtr_s = Xtr_s.fillna(-999)
Xte_s = Xte_s.fillna(-999)

print("Pre-SMOTE training shape:", Xtr_s.shape)
print("Pre-SMOTE class balance:", y_train.value_counts().to_dict())

# 3. Apply SMOTE — TRAINING DATA ONLY
smote = SMOTE(random_state=config.RANDOM_STATE)
Xtr_res, ytr_res = smote.fit_resample(Xtr_s, y_train)

print("\nPost-SMOTE training shape:", Xtr_res.shape)
print("Post-SMOTE class balance:", pd.Series(ytr_res).value_counts().to_dict())

# 4. Train LightGBM WITHOUT scale_pos_weight (SMOTE already balanced it)
model_smote = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=config.RANDOM_STATE,
    n_jobs=-1,
)
model_smote.fit(Xtr_res, ytr_res)

# 5. Evaluate on the UNTOUCHED, real test set
y_prob_smote = model_smote.predict_proba(Xte_s)[:, 1]

print("\n=== LightGBM + SMOTE ===")
evaluate(y_test, y_prob_smote, threshold=config.OPERATING_THRESHOLD)

Pre-SMOTE training shape: (472432, 431)
Pre-SMOTE class balance: {0: 455833, 1: 16599}


d:\Projects\Main Projects\fraud-detection\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "d:\Projects\Main Projects\fraud-detection\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Projects\Main Projects\fraud-detection\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    wi


Post-SMOTE training shape: (911666, 431)
Post-SMOTE class balance: {0: 455833, 1: 455833}
[LightGBM] [Info] Number of positive: 455833, number of negative: 455833
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 4.856379 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 100967
[LightGBM] [Info] Number of data points in the train set: 911666, number of used features: 431
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

=== LightGBM + SMOTE ===
Threshold: 0.5
Precision: 0.7479
Recall:    0.3760
F1:        0.5004
PR-AUC:    0.5377

Confusion matrix [[TN FP] [FN TP]]:
[[113529    515]
 [  2536   1528]]

               precision    recall  f1-score   support

           0     0.9782    0.9955    0.9867    114044
           1     0.7479    0.3760    0.5004      4064

    accuracy                         0.9742    118108
   macro avg     0.8630    0.6857    0.7436    118108
weigh

0.5376701608329381

In [16]:
import joblib
from src import config

config.MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save the winning model + the metadata the API will need
artifact = {
    'model': model,                          # the scale_pos_weight LightGBM
    'cat_cols': cat_cols,                     # which columns are categorical
    'feature_cols': list(X_train.columns),   # exact feature order the model expects
    'threshold': config.OPERATING_THRESHOLD, # 0.50
    'pr_auc': 0.5484,
}

joblib.dump(artifact, config.MODELS_DIR / 'fraud_model.joblib')
print("Saved to:", config.MODELS_DIR / 'fraud_model.joblib')
print("Feature count:", len(artifact['feature_cols']))

Saved to: D:\Projects\Main Projects\fraud-detection\models\fraud_model.joblib
Feature count: 431


In [3]:
import joblib
import pandas as pd
import numpy as np
from src.data import load_processed
from src import config

# Reload data and recreate the exact same time-based split
df = load_processed()
df_sorted = df.sort_values(config.TIME_COL).reset_index(drop=True)
cutoff = int(len(df_sorted) * 0.80)
test_df = df_sorted.iloc[cutoff:]

drop_cols = [config.TARGET, config.ID_COL, config.TIME_COL]
X_test = test_df.drop(columns=drop_cols)
y_test = test_df[config.TARGET]

# Recreate category dtype (must match training)
cat_cols = X_test.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    X_test[col] = X_test[col].astype('category')

# Load the trained model from the saved artifact
artifact = joblib.load(config.MODELS_DIR / "fraud_model.joblib")
model = artifact["model"]

print("Rebuilt: X_test", X_test.shape, "| model loaded")

C:\Users\Dell\AppData\Local\Temp\ipykernel_5000\2684461369.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_test.select_dtypes(include=['object']).columns.tolist()


Rebuilt: X_test (118108, 431) | model loaded


In [8]:
import requests
import numpy as np
import pandas as pd

fraud_idx = list(y_test[y_test == 1].index[:3])
legit_idx = list(y_test[y_test == 0].index[:3])
sample_idx = fraud_idx + legit_idx

# Notebook predictions: predict on a proper SLICE (preserves category dtype)
sample_df = X_test.loc[sample_idx]
notebook_probs = model.predict_proba(sample_df)[:, 1]

for i, idx in enumerate(sample_idx):
    row = X_test.loc[idx]
    notebook_prob = float(notebook_probs[i])

    # Build JSON-safe feature dict for the API
    features = {}
    for k, v in row.items():
        if pd.isna(v):
            features[k] = None
        elif isinstance(v, (np.floating, np.integer)):
            features[k] = float(v)
        else:
            features[k] = str(v)

    resp = requests.post("http://127.0.0.1:8000/predict", json={"features": features})
    api_prob = resp.json()["fraud_probability"]

    label = int(y_test.loc[idx])
    match = "OK" if abs(api_prob - notebook_prob) < 0.01 else "MISMATCH"
    print(f"idx {idx}: actual={label}  notebook={notebook_prob:.4f}  API={api_prob:.4f}  {match}")

idx 472432: actual=1  notebook=0.4774  API=0.4774  OK
idx 472468: actual=1  notebook=0.1692  API=0.1692  OK
idx 472499: actual=1  notebook=0.8455  API=0.8455  OK
idx 472433: actual=0  notebook=0.0003  API=0.0003  OK
idx 472434: actual=0  notebook=0.7284  API=0.7284  OK
idx 472435: actual=0  notebook=0.9247  API=0.9247  OK


In [4]:
import requests

fraud_idx = list(y_test[y_test == 1].index[:3])
legit_idx = list(y_test[y_test == 0].index[:3])
sample_idx = fraud_idx + legit_idx

sample_df = X_test.loc[sample_idx]
notebook_probs = model.predict_proba(sample_df)[:, 1]

for i, idx in enumerate(sample_idx):
    row = X_test.loc[idx]
    notebook_prob = float(notebook_probs[i])

    features = {}
    for k, v in row.items():
        if pd.isna(v):
            features[k] = None
        elif isinstance(v, (np.floating, np.integer)):
            features[k] = float(v)
        else:
            features[k] = str(v)

    resp = requests.post("http://localhost:8000/predict", json={"features": features})
    api_prob = resp.json()["fraud_probability"]

    label = int(y_test.loc[idx])
    match = "OK" if abs(api_prob - notebook_prob) < 0.01 else "MISMATCH"
    print(f"idx {idx}: actual={label}  notebook={notebook_prob:.4f}  container={api_prob:.4f}  {match}")

idx 472432: actual=1  notebook=0.4774  container=0.4774  OK
idx 472468: actual=1  notebook=0.1692  container=0.1692  OK
idx 472499: actual=1  notebook=0.8455  container=0.8455  OK
idx 472433: actual=0  notebook=0.0003  container=0.0003  OK
idx 472434: actual=0  notebook=0.7284  container=0.7284  OK
idx 472435: actual=0  notebook=0.9247  container=0.9247  OK
